In [ ]:
import numpy as np
#sys.path.append('/home/joao/lib/dredge/dredge-python/')
from pathlib import Path
from labdata.schema import *
from labdata import chronic_paper as cp
import matplotlib.pyplot as plt

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
# target dates for chronic timepoint comparison
from datetime import timedelta
COMPARISON_TIMEPOINTS = [timedelta(days=7), timedelta(days=28), timedelta(days=49), timedelta(days=70)]

In [ ]:
chronic_insertions = cp.IBLMatchedInsertion()
chronic_recording_keys = []
for ins in chronic_insertions:
    recordings = (EphysRecording() & ins) * Session()

    for timepoint in COMPARISON_TIMEPOINTS:
        target_datetime = ins['procedure_datetime'] + timepoint
        recordings_with_time_diff = recordings.proj(days_from_target='ABS(TIMESTAMPDIFF(DAY, session_datetime, "{}"))'.format(target_datetime.strftime('%Y-%m-%d %H:%M:%S')))
        key = recordings_with_time_diff.fetch(order_by='days_from_target ASC', as_dict=True, limit=1)[0]
        if key['days_from_target'] > 3:
            print(f"{key['days_from_target']} away from target date {timepoint}")
        chronic_recording_keys.append(key)
chronic_probe_recordings = EphysRecording.proj() & chronic_recording_keys
acute_probe_recordings = EphysRecording.proj() & cp.IBLMatchedInsertion().to_ephys_session()

In [ ]:
# launch spike extraction for all recordings
#all_probe_sessions = EphysRecording.ProbeSetting() & (acute_probe_recordings.fetch('KEY') + chronic_probe_recordings.fetch('KEY'))
all_probe_sessions =  (EphysRecording() & acute_probe_recordings.fetch('KEY') + chronic_probe_recordings.fetch('KEY')) - cp.DredgeSpikeDetection()
labdata_submission_commands = []
t = []
for k in all_probe_sessions.fetch(order_by='subject_name, session_name'):
    dd = f'labdata2 run detect -t aws -a {k["subject_name"]} -s {k["session_name"]} --force-submit'
    t.append(dict(session_name=k['session_name'],
                   subject_name=k['subject_name'],))
    #dd = f'labdata2 run detect -t aws -a {k["subject_name"]} -s {k["session_name"]}'
    labdata_submission_commands.append(dd)
    #os.system(dd) # uncomment to run
labdata_submission_commands = np.unique(labdata_submission_commands)
print(f'There are {len(labdata_submission_commands)} sessions to process.')
print('\n'.join(labdata_submission_commands))

all_probe_session_keys = all_probe_sessions.fetch('KEY')

In [ ]:
# get the probe keys so we can run dredge on each probe-session
acute_session_probe_keys = EphysRecording.ProbeSetting() & (cp.IBLMatchedInsertion().EphysRecording().proj('session_name', 
                                                                                                           'dataset_name',
                                                                                                           chronic_mouse='subject_name',
                                                                                                           chronic_probe_id='probe_id',
                                                                                                           subject_name='matched_subject_name',
                                                                                                           probe_num='matched_probe_num'))
chronic_session_probe_keys = (EphysRecording.ProbeSetting() & chronic_probe_recordings)
chronic_probes = dj.U('subject_name', 'probe_num') & chronic_session_probe_keys # sessions have the same trajectories 

In [ ]:
# now run dredge algorithm on chronic subjects
#subs, probenums = chronic_probes.fetch('subject_name', 'probe_num')
#i = 8
#first_session_per_mouse_probe = chronic_probes.aggr(chronic_session_probe_keys, first_session='MIN(session_name)').fetch(as_dict=True)
#sess = first_session_per_mouse_probe[i]
#cp.DredgeSpikeDetection().plot_raster(sess['subject_name'], sess['first_session'], sess['probe_num'], shank_num=0)

In [ ]:
#mindepth = 0
#maxdepth = 3600
#cp.DredgeMotionEstimate().populate_chronic_subject(sess['subject_name'], sess['probe_num'], dredge_params_id=0, n_workers=4, min_spike_depth=[mindepth], max_spike_depth=[maxdepth])
##(cp.DredgeMotionEstimate() & k).delete()

In [ ]:
# now run dredge on acute subjects
i = 20
sess = acute_session_probe_keys.fetch('subject_name','dataset_name', 'session_name','probe_num', as_dict=True)[i]
print(sess)
cp.DredgeSpikeDetection().plot_raster(sess['subject_name'], sess['session_name'], sess['probe_num'], shank_num=0)

In [ ]:
min_spike_depth = 0
max_spike_depth = 3000
cp.DredgeMotionEstimate().insert_one_session(sess, 0, [min_spike_depth], [max_spike_depth]) 

In [ ]:
DREDGE_PARAMS_ID = 1
# TODO: check that all sessions have been processed
acute_dredge = cp.DredgeMotionEstimate() & acute_probe_recordings & {'dredge_params_id': DREDGE_PARAMS_ID}
chronic_dredge = cp.DredgeMotionEstimate() & chronic_probe_recordings & {'dredge_params_id': DREDGE_PARAMS_ID}
len(chronic_dredge), len(acute_dredge)


In [ ]:
# apply low pass filter to data
from scipy.signal import butter, filtfilt
def lowpass_filter(data, cutoff_freq, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

In [ ]:
from matplotlib import gridspec
spec = gridspec.GridSpec(ncols=1, nrows=2,
                         wspace=.2,
                         hspace=.1,height_ratios=[2,.5])

data = acute_dredge.fetch(as_dict=True)
#data = chronic_dredge.fetch(as_dict=True)
d = data[-4]
print(d['subject_name'], d['session_name'])

fs = np.mean(np.diff(d['time_bin_centers_s']))
filtered = lowpass_filter(d['displacement'], cutoff_freq=.01, fs=fs, order=4)

fig = plt.figure(figsize=(12,8))
ax = fig.add_subplot(spec[0])
plt.gca().spines[['right', 'top']].set_visible(False)

cp.DredgeSpikeDetection().plot_raster(shank_num=0, subject_name=d['subject_name'], session_name=d['session_name'], probe_num=d['probe_num'],
                                      clim=(0,70), cmap='gray_r',rasterized=True)
plt.plot(d['time_bin_centers_s'], d['displacement'].T + d['spatial_bin_centers_um'])
plt.ylabel('Depth along shank (um)')
#plt.gca().spines[['right', 'top']].set_visible(False)

fig.add_subplot(spec[1], sharex=ax)
plt.plot(d['time_bin_centers_s'], d['displacement'].T)
plt.plot(d['time_bin_centers_s'], filtered.T)
plt.ylabel('Drift estimate (um)')
#plt.ylim(-10,10)
plt.ylim(-30, 30)
plt.gca().spines[['right', 'top']].set_visible(False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
#cmap = plt.get_cmap('Set3')
import seaborn as sns

ACUTE_DAYS_OFFSET = -10
N_SAMPS = 30

stats_rows = []
chronic_probe_insertions = dj.U('subject_name','probe_id',) & cp.IBLMatchedInsertion()
cmap = sns.color_palette("hls", len(chronic_probe_insertions))

for i,k in enumerate(chronic_probe_insertions):
    # do chronic insertions
    chronsessions = (chronic_dredge * ProbeInsertion * Session * EphysRecording.ProbeSetting & k).proj('time_bin_centers_s',
                                                                                                       'displacement',
                                                                                                       days_from_insertion='DATEDIFF(session_datetime, procedure_datetime)')
    chronic_datetimes, tss, displacements = chronsessions.fetch('days_from_insertion','time_bin_centers_s','displacement', order_by='days_from_insertion ASC')
    chronic_drifts = []
    for j,(ts, displacement) in enumerate(zip(tss, displacements)):
        fs = np.mean(np.diff(ts))
        filtered = lowpass_filter(displacement, cutoff_freq=.01, fs=fs, order=4)
        #cumulative_drifts = np.sum(np.abs(np.diff(filtered, axis=1)), axis=1)
        cumulative_drifts = np.abs(np.mean(filtered[:,:N_SAMPS], axis=1) - np.mean(filtered[:,-N_SAMPS:], axis=1))
        avg_cumulative_drift = np.mean(cumulative_drifts)
        chronic_drifts.append(avg_cumulative_drift)
        stats_rows.append((i, 1, j, 0, avg_cumulative_drift))
    plt.plot(chronic_datetimes, chronic_drifts, marker='o', color=cmap[i])

    # now do matched acute insertion(s)
    acutesessions = (cp.IBLMatchedInsertion.EphysRecording * (cp.IBLMatchedInsertion.EphysRecording.proj() & chronsessions)).proj('session_name',ttt='probe_id',tt='subject_name',subject_name='matched_subject_name',probe_num='matched_probe_num')
    tss, displacements = (acutesessions * acute_dredge).fetch('time_bin_centers_s','displacement')
    acute_drifts = []
    for j,(ts, displacement) in enumerate(zip(tss, displacements)):
        fs = np.mean(np.diff(ts))
        filtered = lowpass_filter(displacement, cutoff_freq=.01, fs=fs, order=4)
        #cumulative_drifts = np.sum(np.abs(np.diff(filtered, axis=1)), axis=1)
        cumulative_drifts = np.abs(np.mean(filtered[:,:N_SAMPS], axis=1) - np.mean(filtered[:,-N_SAMPS:], axis=1))
        avg_cumulative_drift = np.mean(cumulative_drifts)
        acute_drifts.append(avg_cumulative_drift)
        stats_rows.append((i, 0, 0, j, avg_cumulative_drift))
    plt.scatter(np.random.normal(scale=1, size=len(acute_drifts)) + ACUTE_DAYS_OFFSET, acute_drifts, facecolors='none', marker='o', color=cmap[i])

xticks = [ACUTE_DAYS_OFFSET] + [c.days for c in COMPARISON_TIMEPOINTS]
xticklabels = ['Acute'] + [c.days for c in COMPARISON_TIMEPOINTS]
ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)
ax.set_ylabel('Session drift (um)')
ax.set_xlabel('Days post insertion')

stats_table = pd.DataFrame(stats_rows, columns=['insertion_site','is_chronic','timepoint','session_num','session_drift'])
stats_table.to_csv(Path().resolve().parent / 'stats_tables' / 'acute_chronic_dredge.csv', index=False)